In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração para mostrar números sem notação científica (ex: 1,000,000.00)
pd.options.display.float_format = '{:,.2f}'.format

# --- FUNÇÕES DE LIMPEZA ---
def limpar_valor(valor):
    """Transforma '124M €' em 124000000.0"""
    if pd.isna(valor): return 0
    if isinstance(valor, (int, float)): return valor
    valor = str(valor).replace('€', '').strip()
    if 'M' in valor: return float(valor.replace('M', '')) * 1_000_000
    if 'K' in valor: return float(valor.replace('K', '')) * 1_000
    return float(valor)

def limpar_idade(valor):
    """Transforma '7 fev 1999 (26)' em 26"""
    if pd.isna(valor): return 0
    valor = str(valor)
    if '(' in valor:
        return int(valor.split('(')[-1].replace(')', ''))
    return 0

print("Bibliotecas e funções carregadas!")

Bibliotecas e funções carregadas!


In [2]:
# 1. Carregar Jogadores (Verifique o nome do arquivo)
df = pd.read_csv('jogadores/sofascore_laliga_geral_final_valor_de_mercado_24_25.csv')
df['Liga'] = 'LaLiga'

# 2. Aplicar Limpeza
df['Valor_Numerico'] = df['Valor de mercado'].apply(limpar_valor)
df['Idade_Limpa'] = df['Idade'].apply(limpar_idade)
df['Passe_Numerico'] = df['Acerto no passe %'].apply(lambda x: float(str(x).replace('%','')) if pd.notna(x) else 0)


# Mapeia jogadores para os times corretos da temporada 24/25
correcao_jogadores = {
    'Artem Dovbyk': 'Girona FC',      # Foi pra Roma
    'Aleix García': 'Girona FC',      # Foi pro Leverkusen
    'Savinho': 'Girona FC',           # Foi pro Man City
    'Yan Couto': 'Girona FC',         # Foi pro Dortmund
    'Eric García': 'Girona FC',       # Estava emprestado pelo Barça
    'Pablo Torre': 'Girona FC',       # Estava emprestado pelo Barça

    # --- REAL MADRID ---
    'Toni Kroos': 'Real Madrid',      # Aposentou
    'Nacho Fernández': 'Real Madrid', # Foi pra Arábia
    'Joselu': 'Real Madrid',          # Foi pro Al-Gharafa
    'Kepa Arrizabalaga': 'Real Madrid', # Estava emprestado
    'Kylian Mbappé': 'Real Madrid',   # Chegou agora (Cuidado: stats dele são da Ligue 1, talvez excluir)

    # --- BARCELONA ---
    'Ilkay Gündogan': 'FC Barcelona', # Voltou pro City
    'João Cancelo': 'FC Barcelona',   # Fim de empréstimo
    'João Félix': 'FC Barcelona',     # Fim de empréstimo (foi pro Chelsea)
    'Sergi Roberto': 'FC Barcelona',  # Saiu (Como)
    'Oriol Romeu': 'FC Barcelona',    # Voltou pro Girona (mas jogou 24/25 no Barça)
    'Vitor Roque': 'FC Barcelona',    # Foi pro Betis (mas pertence ao Barça/jogou pouco)

    # --- ATLÉTICO DE MADRID ---
    'Álvaro Morata': 'Atlético Madrid', # Foi pro Milan
    'Memphis Depay': 'Atlético Madrid', # Foi pro Corinthians
    'Stefan Savić': 'Atlético Madrid',  # Saiu
    'Mario Hermoso': 'Atlético Madrid', # Saiu (Roma)
    'Saúl Ñíguez': 'Atlético Madrid',   # Saiu (Sevilla)
    'Arthur Vermeeren': 'Atlético Madrid', # Saiu (Leipzig)

    # --- REAL SOCIEDAD ---
    'Robin Le Normand': 'Real Sociedad', # Foi pro Atlético
    'Mikel Merino': 'Real Sociedad',     # Foi pro Arsenal
    'Kieran Tierney': 'Real Sociedad',   # Fim de empréstimo

    # --- VILLARREAL ---
    'Alexander Sørloth': 'Villarreal',   # Foi pro Atlético (Vice-artilheiro pelo Villarreal)
    'Filip Jørgensen': 'Villarreal',     # Foi pro Chelsea
    'Ben Brereton Díaz': 'Villarreal',   # Saiu
    'Jorge Cuenca': 'Villarreal',        # Saiu

    # --- SEVILLA ---
    'Youssef En-Nesyri': 'Sevilla',      # Foi pro Fenerbahçe
    'Sergio Ramos': 'Sevilla',           # Saiu
    'Marcos Acuña': 'Sevilla',           # Foi pro River
    'Lucas Ocampos': 'Sevilla',          # Saiu

    # --- OUTROS DESTAQUES ---
    'Mason Greenwood': 'Getafe',         # Jogou no Getafe (foi pro Marselha)
    'Jørgen Strand Larsen': 'Celta de Vigo', # Foi pro Wolves
    'Nico Williams': 'Athletic Club',    # Garantia
    'Giorgi Mamardashvili': 'Valencia'   # Garantia (vendido mas ficou emprestado)
}

df['Time_Original'] = df['Time']
df['Time'] = df['Nome'].map(correcao_jogadores).fillna(df['Time'])

print(f"Jogadores carregados: {len(df)}")


Jogadores carregados: 344


In [3]:
# 1. Carregar Tabela
df_tabela = pd.read_csv('tabela/tabela_la_liga_24_25.csv')

# 2. Limpeza de Colunas (Remove espaços extras nos nomes das colunas)
df_tabela.columns = df_tabela.columns.str.strip()

# 3. Renomear para 'Pos' (Garante que o código ache a coluna)
mapa_colunas = {'Posição': 'Pos', '#': 'Pos', 'Rank': 'Pos', 'RK': 'Pos'}
df_tabela.rename(columns=mapa_colunas, inplace=True)

# 4. Tratar Gols (XX:YY) para Gols Pró e Contra
if 'Gols' in df_tabela.columns:
    gols_split = df_tabela['Gols'].str.split(':', expand=True)
    if len(gols_split.columns) == 2:
        df_tabela['Gols_Pro'] = pd.to_numeric(gols_split[0])
        df_tabela['Gols_Contra'] = pd.to_numeric(gols_split[1])


mapa_times_tabela = {
   # Formato Tabela : Formato Jogadores (SofaScore)

    # Real Madrid
    'Real Madrid': 'Real Madrid',
    'R. Madrid': 'Real Madrid',
    
    # Barcelona
    'Barcelona': 'FC Barcelona',
    'FC Barcelona': 'FC Barcelona',
    'Barça': 'FC Barcelona',
    
    # Atlético de Madrid
    'Atlético Madrid': 'Atlético Madrid',
    'Atl. Madrid': 'Atlético Madrid',
    'Atletico Madrid': 'Atlético Madrid',
    'Atletico': 'Atlético Madrid',
    
    # Girona
    'Girona': 'Girona FC',
    'Girona FC': 'Girona FC',
    
    # Athletic Bilbao
    'Athletic Club': 'Athletic Club',
    'Athletic': 'Athletic Club',
    'Athletic Bilbao': 'Athletic Club',
    'Bilbao': 'Athletic Club',
    
    # Real Sociedad
    'Real Sociedad': 'Real Sociedad',
    'R. Sociedad': 'Real Sociedad',
    
    # Betis
    'Real Betis': 'Real Betis',
    'Betis': 'Real Betis',
    'Real Betis Balompié': 'Real Betis',
    
    # Villarreal
    'Villarreal': 'Villarreal',
    'Villarreal CF': 'Villarreal',
    
    # Valencia
    'Valencia': 'Valencia',
    'Valencia CF': 'Valencia',
    
    # Alavés
    'Deportivo Alavés': 'Deportivo Alavés',
    'Alavés': 'Deportivo Alavés',
    'D. Alavés': 'Deportivo Alavés',
    
    # Osasuna
    'Osasuna': 'Osasuna',
    'CA Osasuna': 'Osasuna',
    
    # Getafe
    'Getafe': 'Getafe',
    'Getafe CF': 'Getafe',
    
    # Celta de Vigo
    'Celta Vigo': 'Celta de Vigo',
    'Celta de Vigo': 'Celta de Vigo',
    'RC Celta': 'Celta de Vigo',
    'Celta': 'Celta de Vigo',
    
    # Sevilla
    'Sevilla': 'Sevilla',
    'Sevilla FC': 'Sevilla',
    
    # Mallorca
    'Mallorca': 'RCD Mallorca',
    'RCD Mallorca': 'RCD Mallorca',
    
    # Las Palmas
    'Las Palmas': 'Las Palmas',
    'UD Las Palmas': 'Las Palmas',
    
    # Rayo Vallecano
    'Rayo Vallecano': 'Rayo Vallecano',
    'Rayo': 'Rayo Vallecano',
    
    # Times Rebaixados 23/24 ou Promovidos (dependendo do seu arquivo)
    'Cádiz': 'Cádiz CF',
    'Cadiz': 'Cádiz CF',
    'Almería': 'UD Almería',
    'Granada': 'Granada CF',
    'Leganés': 'CD Leganés',
    'Valladolid': 'Real Valladolid',
    'Espanyol': 'RCD Espanyol'
}
df_tabela['Time'] = df_tabela['Time'].str.strip().replace(mapa_times_tabela)


elenco_stats = df.groupby('Time').agg({
    'Valor_Numerico': 'sum',
    'Média das notas Sofascore': 'mean'
}).reset_index().rename(columns={'Valor_Numerico': 'Valor_Total_Elenco', 'Média das notas Sofascore': 'Media_Sofa_Time'})

# Merge Tabela + Elenco
df_times_final = pd.merge(df_tabela, elenco_stats, on='Time', how='inner')

# Merge Jogadores + Tabela Completa
df_final = pd.merge(df, df_times_final[['Time', 'Pos', 'PTS', 'Media_Sofa_Time', 'Valor_Total_Elenco']], on='Time', how='inner')

print(f"Merge concluído! Total de jogadores com dados completos: {len(df_final)}")

Merge concluído! Total de jogadores com dados completos: 207


In [4]:
# 1. CÁLCULO DO ÍNDICE ATLAS (Performance Individual vs Coletiva)
df_final['Indice_Atlas'] = df_final['Média das notas Sofascore'] - df_final['Media_Sofa_Time']

print("=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===")
display(df_final.sort_values('Indice_Atlas', ascending=False)[['Nome', 'Time', 'Pos', 'Média das notas Sofascore', 'Indice_Atlas']].head(10))

# 2. ANÁLISE DE CONTEXTO (Luxo na Lama)
print("\n=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===")
# Critério: Valor > 20M e Posição > 9
luxo = df_final[
    (df_final['Valor_Numerico'] > 20_000_000) & 
    (df_final['Pos'] > 9)
].sort_values('Valor_Numerico', ascending=False)

if len(luxo) > 0:
    display(luxo[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])
else:
    print("Nenhum jogador encontrado com esses critérios na Bundesliga.")

print("\n=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===")
diamantes = df_final[
    (df_final['Média das notas Sofascore'] > 7.15) & 
    (df_final['Pos'] > 9)
].sort_values('Média das notas Sofascore', ascending=False)

display(diamantes[['Nome', 'Time', 'Pos', 'Valor de mercado', 'Média das notas Sofascore']])

=== 🏋️ ÍNDICE ATLAS: QUEM CARREGA O TIME? ===


,Nome,Time,Pos,Média das notas Sofascore,Indice_Atlas
15,Isco,Real Betis,6,7.54,0.55
0,Kylian Mbappé,Real Madrid,2,7.70,0.52
27,Alejandro Baena,Atlético Madrid,3,7.48,0.47
16,Giovani Lo Celso,Real Betis,6,7.41,0.42
58,Dani Raba,Valencia,12,7.23,0.37
43,Sergio Herrera,Osasuna,9,7.24,0.36
105,Nahuel Tenaglia,Deportivo Alavés,15,7.14,0.31
116,Sergio Gómez,Real Sociedad,11,7.11,0.29
1,Federico Valverde,Real Madrid,2,7.47,0.29
151,Luis Milla,Getafe,13,7.10,0.27



=== 💸 LUXO DESPERDIÇADO (Jogadores Caros em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
120,Robin Le Normand,Real Sociedad,11,33M €,6.90
117,Takefusa Kubo,Real Sociedad,11,31M €,7.07
64,Giorgi Mamardashvili,Valencia,12,29M €,6.89
69,Javier Guerra,Valencia,12,26M €,6.77
169,Eric García,Girona FC,16,24M €,6.99
124,Mikel Oyarzabal,Real Sociedad,11,21M €,6.81



=== 💎 DIAMANTES (Jogadores de Elite em Times da Parte de Baixo) ===


,Nome,Time,Pos,Valor de mercado,Média das notas Sofascore
58,Dani Raba,Valencia,12,2.6M €,7.23


In [5]:
def ficha_jogador(parte_do_nome):
    """
    Busca um jogador pelo nome (ou parte dele) e mostra todas as colunas.
    """
    dataframe_alvo = df 
    
    mask = dataframe_alvo['Nome'].str.lower().str.contains(parte_do_nome.lower(), na=False)
    resultado = dataframe_alvo[mask]
    
    if len(resultado) == 0:
        return f"❌ Nenhum jogador encontrado com o nome '{parte_do_nome}'."

    return resultado.T

nome_para_buscar = input("Digite o nome (ou parte do nome) do jogador: ")

display(ficha_jogador(nome_para_buscar))

,4
Link Jogador,https://www.sofascore.com/pt/football/player/i...
Time,Real Betis
Valor de mercado,5.2M €
Idade,21 de abril de 1992 (33)
Pontos fortes,Sem pontos fortes notáveis
Pontos fracos,Sem pontos fracos notáveis
Posição,Midfielder
Gols,9
Gols esperados (xG),8.62
Dribles certos,27


In [6]:
# Preparando dados para o gráfico
grafico_df = df_times_final.copy()

# Tratamento para evitar erro no gráfico (Divisão por zero ou NaN)
grafico_df['PTS'] = pd.to_numeric(grafico_df['PTS'], errors='coerce').fillna(0)
grafico_df['Valor_Total_Elenco'] = pd.to_numeric(grafico_df['Valor_Total_Elenco'], errors='coerce').fillna(0)
pts_ajustado = grafico_df['PTS'].replace(0, 1) # Evita dividir por zero

grafico_df['Custo_Por_Ponto'] = grafico_df['Valor_Total_Elenco'] / pts_ajustado

fig = px.scatter(grafico_df, 
                 x='Valor_Total_Elenco', 
                 y='PTS', 
                 text='Time', 
                 size='Custo_Por_Ponto', 
                 color='Pos',
                 color_continuous_scale='Bluered_r', # Azul = Topo, Vermelho = Baixo
                 hover_data=['Custo_Por_Ponto'],
                 title='Eficiência Financeira Bundesliga: Valor do Elenco x Pontos',
                 labels={'Valor_Total_Elenco': 'Investimento Total (€)', 'PTS': 'Pontos Conquistados'})

fig.update_traces(textposition='top center')

fig.update_yaxes(range=[0, 100], autorange=False, dtick=10)
fig.show()

In [7]:
def scouting_laliga(posicao=None, orcamento_max=None, atributo_obrigatorio=None):
    res = df_final.copy()
    
    if posicao:
        res = res[res['Posição'].str.contains(posicao, case=False, na=False)]
    
    if orcamento_max:
        valor = float(orcamento_max) * 1_000_000
        res = res[res['Valor_Numerico'] <= valor]
        
    if atributo_obrigatorio:
        res = res[res['Pontos fortes'].str.contains(atributo_obrigatorio, case=False, na=False)]
        
    cols = ['Nome', 'Time', 'Idade_Limpa', 'Valor de mercado', 'Média das notas Sofascore', 'Pontos fortes']
    return res.sort_values('Média das notas Sofascore', ascending=False)[cols].head(10)


print("🔍 Resultado Scouting :")
display(scouting_laliga(posicao='Midfielder', orcamento_max=50))

🔍 Resultado Scouting :


,Nome,Time,Idade_Limpa,Valor de mercado,Média das notas Sofascore,Pontos fortes
15,Isco,Real Betis,33,5.2M €,7.54,Sem pontos fortes notáveis
16,Giovani Lo Celso,Real Betis,29,14.6M €,7.41,Cobranças direta de falta; Passes longos; Passe
6,Eduardo Camavinga,Real Madrid,23,48M €,7.16,Duelos no chão; Passes longos; Desarme
45,Rubén García,Osasuna,32,2.1M €,7.13,Cobranças direta de falta; Passes longos; Armador
7,Dani Ceballos,Real Madrid,29,8.4M €,7.13,Armador; Duelos no chão; Passes longos
116,Sergio Gómez,Real Sociedad,25,14.6M €,7.11,Cobranças direta de falta; Passe; Armador
151,Luis Milla,Getafe,31,3.6M €,7.10,Alta pressão; Armador; Duelos no chão
117,Takefusa Kubo,Real Sociedad,24,31M €,7.07,Alta pressão; Armador
167,Viktor Tsygankov,Girona FC,27,15.7M €,7.07,Armador
181,Joan Jordán,Sevilla,31,2.4M €,7.06,Sem pontos fortes notáveis
